## 1. Kütüphaneler

In [1]:
import os
from pathlib import Path
import pandas as pd

## 2. Çalışma Dizini Ayarı
Notebook `notebooks/` klasöründen çalıştırıldığında proje köküne dönmek için dizin ayarlanır.

In [2]:
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 3.Veri Klasörü Ayarları

In [3]:
os.listdir("data")

['amazon_reviews_us_Electronics_v1_00.tsv',
 'amazon_reviews_us_Mobile_Electronics_v1_00.tsv',
 'amazon_reviews_us_Video_Games_v1_00.tsv',
 '.ipynb_checkpoints']

In [4]:

import pandas as pd

file_path = "data/amazon_reviews_us_Video_Games_v1_00.tsv"

df = pd.read_csv(
    file_path,
    sep="\t",
    nrows=5,
    engine="python"
)

df.head()

,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,12039526,RTIS3L2M1F5SM,B001CXYMFS,737716809,Thrustmaster T-Flight Hotas X Flight Stick,Video Games,5,0,0,N,Y,an amazing joystick. I especially love that yo...,"Used this for Elite Dangerous on my mac, an am...",2015-08-31
1,US,9636577,R1ZV7R40OLHKD,B00M920ND6,569686175,Tonsee 6 buttons Wireless Optical Silent Gamin...,Video Games,5,0,0,N,Y,Definitely a silent mouse... Not a single clic...,"Loved it, I didn't even realise it was a gami...",2015-08-31
2,US,2331478,R3BH071QLH8QMC,B0029CSOD2,98937668,Hidden Mysteries: Titanic Secrets of the Fatef...,Video Games,1,0,1,N,Y,One Star,poor quality work and not as it is advertised.,2015-08-31
3,US,52495923,R127K9NTSXA2YH,B00GOOSV98,23143350,GelTabz Performance Thumb Grips - PlayStation ...,Video Games,3,0,0,N,Y,"good, but could be bettee","nice, but tend to slip away from stick in inte...",2015-08-31
4,US,14533949,R32ZWUXDJPW27Q,B00Y074JOM,821342511,Zero Suit Samus amiibo - Japan Import (Super S...,Video Games,4,0,0,N,Y,Great but flawed.,"Great amiibo, great for collecting. Quality ma...",2015-08-31


## 4. Veri Yükleme
Her üç kategori için ilk 200.000 satır okunur. Hatalı satırlar atlanır.

In [5]:
import pandas as pd
import csv

def read_amazon_tsv(path, nrows=200000):
    return pd.read_csv(
        path,
        sep="\t",
        nrows=nrows,
        quoting=csv.QUOTE_NONE,
        on_bad_lines="skip",
        engine="python"
    )

electronics = read_amazon_tsv("data/amazon_reviews_us_Electronics_v1_00.tsv")
mobile = read_amazon_tsv("data/amazon_reviews_us_Mobile_Electronics_v1_00.tsv")
video_games = read_amazon_tsv("data/amazon_reviews_us_Video_Games_v1_00.tsv")

## 5. Keşifsel Veri Analizi
Her dataset için eksik değer oranı, yıldız dağılımı ve doğrulanmış satın alma oranı incelenir.

In [6]:
def eda_summary(df, name):
    print(f"\n{'='*40}")
    print(f"DATASET: {name} | Shape: {df.shape}")
    print(f"{'='*40}")
    
    # Missing values
    missing = df.isna().mean().mul(100).round(2)
    print("\n--- Missing % ---")
    print(missing[missing > 0])
    
    # Star rating dağılımı
    print("\n--- Star Rating % ---")
    print(df["star_rating"].value_counts(normalize=True).mul(100).round(2).sort_index())
    
    # Verified purchase
    print("\n--- Verified Purchase % ---")
    print(df["verified_purchase"].value_counts(normalize=True).mul(100).round(2))

eda_summary(electronics, "Electronics")
eda_summary(mobile, "Mobile Electronics")
eda_summary(video_games, "Video Games")


DATASET: Electronics | Shape: (200000, 15)

--- Missing % ---
review_body    0.02
dtype: float64

--- Star Rating % ---
star_rating
1    11.54
2     5.30
3     6.98
4    14.93
5    61.25
Name: proportion, dtype: float64

--- Verified Purchase % ---
verified_purchase
Y    91.91
N     8.09
Name: proportion, dtype: float64

DATASET: Mobile Electronics | Shape: (104975, 15)

--- Missing % ---
Series([], dtype: float64)

--- Star Rating % ---
star_rating
1    16.75
2     6.96
3     9.27
4    17.23
5    49.78
Name: proportion, dtype: float64

--- Verified Purchase % ---
verified_purchase
Y    84.31
N    15.69
Name: proportion, dtype: float64

DATASET: Video Games | Shape: (200000, 15)

--- Missing % ---
review_body    0.02
dtype: float64

--- Star Rating % ---
star_rating
1     9.79
2     4.23
3     6.90
4    13.47
5    65.62
Name: proportion, dtype: float64

--- Verified Purchase % ---
verified_purchase
Y    90.3
N     9.7
Name: proportion, dtype: float64


## 6. Dataset Karşılaştırması
Üç kategori tek tabloda birleştirilip yıldız dağılımları karşılaştırılır.

In [7]:
for df, name in [(electronics, "Electronics"), (mobile, "Mobile"), (video_games, "Video Games")]:
    df["dataset"] = name

combined = pd.concat([electronics, mobile, video_games], ignore_index=True)

# Tek tabloda karşılaştır
combined.groupby("dataset")["star_rating"].agg(["mean", "median", "count"]).round(2)

,mean,median,count
dataset,,,
Electronics,4.09,5.0,200000
Mobile,3.76,4.0,104975
Video Games,4.21,5.0,200000


## 7. Temiz Veri Seti Oluşturma
Model için gerekli sütunlar seçilir. Eksik ve çok kısa yorumlar çıkarılır. Her kategoriden 50.000 satır örneklenir.

In [8]:
COLS = ["review_headline", "review_body", "star_rating", "verified_purchase", "helpful_votes", "total_votes", "dataset"]

combined_clean = (
    combined[COLS]
    .dropna(subset=["review_body"])
    .query("review_body.str.split().str.len() >= 5")
    .groupby("dataset", group_keys=False)
    .apply(lambda x: x.sample(50_000, random_state=42), include_groups=False)
    .reset_index(drop=True)
)

print(combined_clean.shape)
combined_clean.head()

(150000, 6)


,review_headline,review_body,star_rating,verified_purchase,helpful_votes,total_votes
0,"Great product, great price!",Got here on time and it works great. It turns ...,5,Y,0,0
1,Five Stars,"Really nice quality, and the color is nice..",5,Y,0,0
2,Five Stars,3 years installed. 55&#34; TV with no sag.,5,Y,0,0
3,Not sure what happened but after 3 months of o...,Not sure what happened but after 3 months of o...,1,Y,0,13
4,Amazing sound quality and build quality,"Amazing sound quality and build quality, this ...",5,Y,0,0


Veri Yapısı

3 kategori, her birinden 50k satır → toplam 150k satır

Kullanılacak 7 sütun belirlendi



review_headline — yorum başlığı

review_body — yorum metni (modelin ana girdisi)

star_rating — yıldız puanı (label üretiminde sinyal)

verified_purchase — doğrulanmış satın alma

helpful_votes — faydalı oy sayısı

total_votes — toplam oy sayısı

dataset — hangi kategoriden geldiği (Electronics, Mobile, Video Games)

Yıldız Dağılımı

3 kategoride de 5 yıldız baskın (~%60+)

1-2 yıldız azınlıkta → model eğitiminde class imbalance sorunu olacak

Verified Purchase

Verified yorumlarda 5 yıldız biraz daha fazla

Non-verified yorumlarda 1 yıldız biraz daha fazla

Fark büyük değil ama sinyal olarak kullanılabilir

Metin Kalitesi

5 kelimeden kısa yorumlar filtrelendi
review_body eksik olanlar düşürüldü

Sonraki adım için bunlar önemli:

Class imbalance var → labeling aşamasında bunu dengelemek gerekecek

star_rating + review_body birlikte label üretiminde sinyal olacak

helpful_votes / total_votes kaliteli yorum filtresi olarak kullanılabilir

In [9]:
combined_clean.to_csv("data/combined_clean.csv", index=False)